# Definitivo — dati FIRMS, segmentazione, lag e target

Pipeline canonica: usa tutto lo storico grezzo per identificare segmenti continui e crea un pannello campionato riproducibile, trattabile e privo di falsi zeri sui vuoti temporali.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

RANDOM_STATE = 42
GRID_SIZE = 0.1
MIN_SEGMENT_DAYS = 28
MAX_CELLS_PER_SEGMENT = 5_000  # campione riproducibile: evita un pannello globale >70M righe.
SOURCES = [
    Path("../dataset_incendi_FIRMS.csv"),
    Path("../matteo/dataset_incendi_FIRMS.csv"),
]
OUTPUT_DIR = Path("output_definitivo")
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_PANEL = OUTPUT_DIR / "incendi_daily_segmentato_sample_v3.csv"
USECOLS = ["latitude", "longitude", "acq_date", "frp"]

for source in SOURCES:
    if not source.exists():
        raise FileNotFoundError(source.resolve())

def chunks():
    for source in SOURCES:
        for part in pd.read_csv(source, usecols=USECOLS, chunksize=500_000,
                                dtype={"latitude": "float32", "longitude": "float32", "frp": "float32"}):
            part["date"] = pd.to_datetime(part.pop("acq_date"), errors="coerce").dt.normalize()
            part = part.dropna(subset=["latitude", "longitude", "frp", "date"])
            part["lat_cell"] = (np.floor(part["latitude"] / GRID_SIZE) * GRID_SIZE).round(4).astype("float32")
            part["lon_cell"] = (np.floor(part["longitude"] / GRID_SIZE) * GRID_SIZE).round(4).astype("float32")
            yield part

# I giorni senza alcun record nell'intera sorgente interrompono il segmento: mai interpretati come zero-incendi.
daily_parts = []
for part in chunks():
    daily_parts.append(part.groupby("date", observed=True).size())
daily_observed = pd.concat(daily_parts).groupby(level=0).sum().sort_index()
all_dates = pd.date_range(daily_observed.index.min(), daily_observed.index.max(), freq="D")
segment_by_date, segment_rows = {}, []
start = previous = daily_observed.index[0]
segment_id = 0
for date in daily_observed.index[1:]:
    if date - previous > pd.Timedelta(days=1):
        run = pd.date_range(start, previous, freq="D")
        segment_rows.append({"segment_id": segment_id, "start": start, "end": previous, "days": len(run)})
        segment_by_date.update({d: segment_id for d in run})
        segment_id += 1
        start = date
    previous = date
run = pd.date_range(start, previous, freq="D")
segment_rows.append({"segment_id": segment_id, "start": start, "end": previous, "days": len(run)})
segment_by_date.update({d: segment_id for d in run})

segments = pd.DataFrame(segment_rows)
segments["usable_for_model"] = segments["days"].ge(MIN_SEGMENT_DAYS)
segments["reason"] = np.where(segments.usable_for_model, "segmento_continuo_utilizzabile", "troppo_corto_escluso")
segments.to_csv(OUTPUT_DIR / "segmenti_temporali_v3.csv", index=False)
usable_ids = set(segments.loc[segments.usable_for_model, "segment_id"])
print(segments.to_string(index=False))

# Conta attività per cella e segmento, poi estrae un campione uniforme di celle attive per segmento.
cell_parts = []
for part in chunks():
    part["segment_id"] = part.date.map(segment_by_date)
    part = part[part.segment_id.isin(usable_ids)]
    cell_parts.append(part.groupby(["segment_id", "lat_cell", "lon_cell"], observed=True).size().rename("detections"))
cell_counts = pd.concat(cell_parts).groupby(level=[0, 1, 2]).sum().reset_index()
rng = np.random.default_rng(RANDOM_STATE)
sampled_parts = []
for current_segment, group in cell_counts.groupby("segment_id", observed=True):
    chosen = rng.choice(group.index.to_numpy(), size=min(MAX_CELLS_PER_SEGMENT, len(group)), replace=False)
    sampled_parts.append(group.loc[chosen])
sampled_cells = pd.concat(sampled_parts, ignore_index=True)
sampled_cells.to_csv(OUTPUT_DIR / "campione_celle_v3.csv", index=False)
print(sampled_cells.groupby("segment_id").size().rename("celle_campionate").to_dict())

# Aggregazione quotidiana soltanto per le celle campionate; le assenze interne ai segmenti diventano zero documentati.
daily_parts = []
sample_keys = sampled_cells[["segment_id", "lat_cell", "lon_cell"]]
for part in chunks():
    part["segment_id"] = part.date.map(segment_by_date)
    part = part.merge(sample_keys, on=["segment_id", "lat_cell", "lon_cell"], how="inner")
    if len(part):
        daily_parts.append(part.groupby(["segment_id", "lat_cell", "lon_cell", "date"], as_index=False, observed=True)
                           .agg(detection_count=("frp", "size"), daily_frp_mean=("frp", "mean"),
                                daily_frp_sum=("frp", "sum"), daily_frp_max=("frp", "max")))
daily = pd.concat(daily_parts, ignore_index=True).groupby(["segment_id", "lat_cell", "lon_cell", "date"], as_index=False, observed=True).agg(
    detection_count=("detection_count", "sum"), daily_frp_sum=("daily_frp_sum", "sum"),
    daily_frp_max=("daily_frp_max", "max"))
daily["daily_frp_mean"] = daily["daily_frp_sum"] / daily["detection_count"]

# Materializzazione per segmento: evita la falsa continuità fra intervalli assenti e mantiene la memoria limitata.
OUTPUT_PANEL.unlink(missing_ok=True)
first = True
for row in segments[segments.usable_for_model].itertuples(index=False):
    dates = pd.DataFrame({"date": pd.date_range(row.start, row.end, freq="D")})
    cells = sampled_cells.loc[sampled_cells.segment_id.eq(row.segment_id), ["segment_id", "lat_cell", "lon_cell"]]
    panel = cells.merge(dates, how="cross").merge(daily.loc[daily.segment_id.eq(row.segment_id)], on=["segment_id", "lat_cell", "lon_cell", "date"], how="left", validate="one_to_one")
    for col in ["detection_count", "daily_frp_mean", "daily_frp_sum", "daily_frp_max"]:
        panel[col] = panel[col].fillna(0).astype("float32")
    panel["active_fire_day"] = panel.detection_count.gt(0).astype("int8")
    panel["segment_start"], panel["segment_end"] = pd.Timestamp(row.start), pd.Timestamp(row.end)
    panel.to_csv(OUTPUT_PANEL, index=False, mode="w" if first else "a", header=first)
    first = False

profile = pd.DataFrame([{
    "raw_rows_estimated": int(daily_observed.sum()), "observed_days": len(daily_observed),
    "calendar_days": len(all_dates), "missing_days": len(all_dates) - len(daily_observed),
    "usable_segments": int(segments.usable_for_model.sum()), "sample_cells_per_segment_max": MAX_CELLS_PER_SEGMENT,
    "grid_size_degrees": GRID_SIZE, "panel_rows": sum(segments.loc[segments.usable_for_model, "days"] * MAX_CELLS_PER_SEGMENT),
    "note": "Campione di celle attive; non stimare prevalenze globali senza pesi di inclusione.",
}])
profile.to_csv(OUTPUT_DIR / "profilo_storico_segmentato_v3.csv", index=False)
with open(OUTPUT_DIR / "configurazione_v3.json", "w") as fp:
    json.dump({"random_state": RANDOM_STATE, "grid_size": GRID_SIZE, "min_segment_days": MIN_SEGMENT_DAYS,
               "max_cells_per_segment": MAX_CELLS_PER_SEGMENT, "sources": [str(p) for p in SOURCES]}, fp, indent=2)
print({"panel": str(OUTPUT_PANEL), "rows": sum(1 for _ in open(OUTPUT_PANEL)) - 1, "segments": int(segments.usable_for_model.sum())})
